# Chapter 15: AI Readiness Assessment Tool

**From Enterprise Architect to AI Enterprise Architect**

A structured assessment tool for evaluating your enterprise's readiness to adopt AI.
Input your current state across five dimensions — data maturity, infrastructure,
team skills, governance, and organizational readiness — and generate a prioritized
AI readiness report with recommended next steps.

**Key Insight:** Honest self-assessment prevents the most common AI adoption failures.
Organizations that skip this step build on sand.

---

## Setup

This notebook is self-contained and does not require any external APIs.
It uses only Python standard library and tabulate for display.

In [ ]:
# Install dependencies
!pip install -q tabulate

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from tabulate import tabulate
import json

## Define the Assessment Framework

The assessment covers **5 dimensions**, each with **4 sub-questions**.
Every sub-question is scored on a 1-5 scale:

| Score | Level | Description |
|---|---|---|
| 1 | Not Started | No capability or awareness |
| 2 | Ad Hoc | Informal, inconsistent efforts |
| 3 | Developing | Defined processes, partial adoption |
| 4 | Established | Consistent, measured, improving |
| 5 | Mature | Optimized, automated, industry-leading |

In [ ]:
# Assessment framework: dimensions and sub-questions with scoring rubrics
ASSESSMENT_FRAMEWORK = {
    "Data Maturity": {
        "weight": 0.25,  # Data is the foundation — weighted highest
        "questions": {
            "Data Quality": {
                "prompt": "How clean, accurate, and complete is your core business data?",
                "rubric": {
                    1: "No data quality processes; frequent data issues",
                    2: "Ad hoc data cleaning; known quality problems",
                    3: "Defined quality rules; regular monitoring for key datasets",
                    4: "Automated quality checks; dashboards; < 5% issues",
                    5: "Real-time quality monitoring; self-healing pipelines; < 1% issues",
                },
            },
            "Data Accessibility": {
                "prompt": "How easily can teams access the data they need?",
                "rubric": {
                    1: "Data locked in silos; manual extraction only",
                    2: "Some shared databases; access requires IT tickets",
                    3: "Data warehouse in place; self-service for common queries",
                    4: "Data lake/lakehouse; cross-team access with governance",
                    5: "Data mesh / federated access; real-time availability; APIs",
                },
            },
            "Data Cataloging": {
                "prompt": "Is your data cataloged, documented, and discoverable?",
                "rubric": {
                    1: "No catalog; tribal knowledge only",
                    2: "Spreadsheets or wikis with partial documentation",
                    3: "Data catalog tool deployed; key datasets documented",
                    4: "Comprehensive catalog; lineage tracking; active usage",
                    5: "Automated cataloging; full lineage; impact analysis",
                },
            },
            "Data Governance": {
                "prompt": "Do you have data ownership, policies, and stewardship?",
                "rubric": {
                    1: "No formal ownership or policies",
                    2: "Some owners identified; policies exist but not enforced",
                    3: "Data stewards assigned; policies documented and partly enforced",
                    4: "Active governance council; enforced policies; audit trails",
                    5: "Automated policy enforcement; continuous compliance; mature stewardship",
                },
            },
        },
    },
    "Technical Infrastructure": {
        "weight": 0.20,
        "questions": {
            "Cloud Adoption": {
                "prompt": "What is your level of cloud adoption and maturity?",
                "rubric": {
                    1: "All on-premises; no cloud strategy",
                    2: "Experimenting with cloud; a few workloads migrated",
                    3: "Hybrid cloud; core services migrating; IaC for new workloads",
                    4: "Cloud-first policy; most workloads in cloud; well-architected",
                    5: "Cloud-native; multi-cloud; auto-scaling; cost-optimized",
                },
            },
            "API Availability": {
                "prompt": "Are your core systems accessible via APIs?",
                "rubric": {
                    1: "No APIs; batch file transfers; FTP",
                    2: "A few internal APIs; no standards",
                    3: "API gateway in place; key systems exposed; documentation exists",
                    4: "API-first design; versioned APIs; developer portal",
                    5: "Full API economy; event-driven; GraphQL/REST standards",
                },
            },
            "CI/CD Maturity": {
                "prompt": "How mature are your deployment and delivery pipelines?",
                "rubric": {
                    1: "Manual deployments; no version control for infra",
                    2: "Basic CI; manual deployments; some automation",
                    3: "CI/CD pipelines for most services; staging environments",
                    4: "Automated testing + deployment; feature flags; rollbacks",
                    5: "GitOps; canary deployments; automated rollbacks; full observability",
                },
            },
            "Compute Resources": {
                "prompt": "Do you have access to GPU/ML compute when needed?",
                "rubric": {
                    1: "No GPU access; standard office hardware only",
                    2: "Can request cloud GPUs; no established process",
                    3: "GPU instances available; used for specific projects",
                    4: "ML platform with managed compute; auto-scaling",
                    5: "Dedicated ML infrastructure; spot/reserved instances; cost tracking",
                },
            },
        },
    },
    "Team Skills": {
        "weight": 0.20,
        "questions": {
            "ML/AI Knowledge": {
                "prompt": "Does your team understand ML/AI concepts and limitations?",
                "rubric": {
                    1: "No ML/AI knowledge in the team",
                    2: "A few individuals self-learning; no formal training",
                    3: "Training programs in place; some team members certified",
                    4: "Dedicated ML engineers; cross-functional AI literacy",
                    5: "Deep expertise; publishing research; contributing to open source",
                },
            },
            "Python Proficiency": {
                "prompt": "How proficient is your engineering team with Python and ML tooling?",
                "rubric": {
                    1: "No Python skills; Java/C# only",
                    2: "Basic scripting; a few Python users",
                    3: "Python used in production; pandas/numpy familiarity",
                    4: "Strong Python team; ML libraries; notebook workflows",
                    5: "Expert-level; custom libraries; mentoring others",
                },
            },
            "Data Engineering": {
                "prompt": "Can your team build and maintain data pipelines?",
                "rubric": {
                    1: "No data engineering capability",
                    2: "Basic ETL scripts; fragile pipelines",
                    3: "Orchestrated pipelines (Airflow/dbt); monitoring",
                    4: "Robust pipelines; data contracts; SLAs met",
                    5: "Real-time streaming; self-service pipelines; data mesh",
                },
            },
            "Prompt Engineering": {
                "prompt": "Does your team know how to design and evaluate LLM prompts?",
                "rubric": {
                    1: "No experience with LLMs or prompt design",
                    2: "Casual ChatGPT users; no systematic approach",
                    3: "Structured prompts; some evaluation; RAG awareness",
                    4: "Prompt libraries; A/B testing; systematic evaluation",
                    5: "Advanced techniques; fine-tuning; prompt optimization; evals",
                },
            },
        },
    },
    "Governance": {
        "weight": 0.20,
        "questions": {
            "Data Governance Framework": {
                "prompt": "Do you have a formal data governance framework?",
                "rubric": {
                    1: "No data governance framework",
                    2: "Informal guidelines; not consistently applied",
                    3: "Documented framework; governance roles defined",
                    4: "Active enforcement; regular audits; metrics tracked",
                    5: "Automated compliance; real-time monitoring; continuous improvement",
                },
            },
            "Security Policies": {
                "prompt": "Are security policies defined for AI/ML workloads?",
                "rubric": {
                    1: "No AI-specific security policies",
                    2: "General IT security; no AI-specific controls",
                    3: "AI security guidelines defined; model access controls",
                    4: "Enforced AI security; data classification; prompt injection defenses",
                    5: "AI red teaming; adversarial testing; zero-trust for models",
                },
            },
            "Compliance Processes": {
                "prompt": "Can you demonstrate AI compliance for regulators?",
                "rubric": {
                    1: "No compliance processes for AI",
                    2: "Aware of requirements; no processes yet",
                    3: "Compliance checklist exists; manual reviews",
                    4: "Automated compliance checks; audit trails; model cards",
                    5: "Continuous compliance; regulatory reporting; bias audits",
                },
            },
            "Risk Management": {
                "prompt": "Do you have AI-specific risk assessment and mitigation?",
                "rubric": {
                    1: "No AI risk management",
                    2: "Risks acknowledged but not formally tracked",
                    3: "Risk register includes AI; mitigation plans for top risks",
                    4: "Quantified risk scoring; regular reviews; incident playbooks",
                    5: "Real-time risk monitoring; automated circuit breakers; mature response",
                },
            },
        },
    },
    "Organizational Readiness": {
        "weight": 0.15,
        "questions": {
            "Executive Sponsorship": {
                "prompt": "Is there active executive sponsorship for AI initiatives?",
                "rubric": {
                    1: "No executive interest or awareness",
                    2: "Interest but no budget or mandate",
                    3: "Sponsor identified; budget allocated for pilots",
                    4: "C-level champion; AI in strategic plan; dedicated budget",
                    5: "Board-level priority; AI embedded in corporate strategy",
                },
            },
            "Change Management": {
                "prompt": "Do you have change management capability for AI adoption?",
                "rubric": {
                    1: "No change management; technology-push only",
                    2: "Basic communications; no structured approach",
                    3: "Change management plan; stakeholder mapping; training",
                    4: "Change champions network; feedback loops; adoption metrics",
                    5: "Mature change practice; continuous adoption; culture of experimentation",
                },
            },
            "AI Strategy": {
                "prompt": "Is there a documented AI strategy aligned with business goals?",
                "rubric": {
                    1: "No AI strategy",
                    2: "Informal ideas; no documented strategy",
                    3: "Documented strategy; aligned with 1-2 business goals",
                    4: "Comprehensive strategy; use case portfolio; roadmap",
                    5: "AI strategy integrated with enterprise strategy; measured outcomes",
                },
            },
            "Innovation Culture": {
                "prompt": "Does your culture support experimentation and learning from failure?",
                "rubric": {
                    1: "Risk-averse; failure is punished",
                    2: "Some tolerance for experimentation in IT",
                    3: "Innovation time / hackathons; safe-to-fail pilots",
                    4: "Experimentation embedded in process; learning celebrated",
                    5: "Innovation lab; venture model; rapid prototyping culture",
                },
            },
        },
    },
}

print(f"Assessment framework loaded: {len(ASSESSMENT_FRAMEWORK)} dimensions, "
      f"{sum(len(d['questions']) for d in ASSESSMENT_FRAMEWORK.values())} questions")

## The Assessment Class

The `AIReadinessAssessment` class takes scores for each sub-question and
generates a comprehensive readiness report including:

- Overall weighted readiness score
- Readiness level classification
- Radar chart data (as a text table)
- Top strengths and gaps
- Prioritized action plan
- Recommended integration pattern from Chapter 9

In [ ]:
class AIReadinessAssessment:
    """AI readiness assessment tool for enterprise architects."""

    LEVELS = {
        (1.0, 2.0): "Exploring",
        (2.0, 3.0): "Developing",
        (3.0, 4.0): "Established",
        (4.0, 5.1): "Leading",  # 5.1 to include score of 5.0
    }

    # Action plan templates keyed by dimension and score range
    ACTION_PLANS = {
        "Data Maturity": {
            "low": [
                "Conduct a data audit to inventory all data sources and assess quality",
                "Assign data owners for top 10 business-critical datasets",
                "Deploy a data catalog tool (e.g., DataHub, Amundsen) within 3 months",
            ],
            "high": [
                "Implement automated data quality monitoring with alerting",
                "Build self-service data access with proper governance guardrails",
            ],
        },
        "Technical Infrastructure": {
            "low": [
                "Develop cloud migration strategy for AI workloads within 2 months",
                "Expose 5 core systems via REST APIs in the next quarter",
                "Set up CI/CD pipelines for at least the AI/ML codebase",
            ],
            "high": [
                "Deploy an ML platform (e.g., MLflow, SageMaker, Vertex AI)",
                "Implement GPU auto-scaling and cost tracking for ML workloads",
            ],
        },
        "Team Skills": {
            "low": [
                "Launch AI/ML training program — start with prompt engineering workshops",
                "Hire or contract 1-2 ML engineers to seed the team",
                "Set up a shared Jupyter/Colab environment for experimentation",
            ],
            "high": [
                "Create an internal AI community of practice",
                "Build a prompt library and evaluation framework",
            ],
        },
        "Governance": {
            "low": [
                "Draft an AI acceptable use policy within 1 month",
                "Create an AI risk register with top 10 risks and mitigations",
                "Define model approval process and compliance checklist",
            ],
            "high": [
                "Implement automated model monitoring and bias detection",
                "Establish AI red-teaming practice for high-risk models",
            ],
        },
        "Organizational Readiness": {
            "low": [
                "Identify and secure executive sponsor — prepare ROI case (see Ch13)",
                "Develop AI strategy document aligned with 2-3 business priorities",
                "Plan a low-risk AI pilot to build organizational confidence",
            ],
            "high": [
                "Scale change management: train change champions in each business unit",
                "Embed AI metrics into business KPIs and executive dashboards",
            ],
        },
    }

    def __init__(self, organization_name: str, scores: Dict[str, Dict[str, int]]):
        """
        Args:
            organization_name: Name of the organization being assessed.
            scores: Nested dict of {dimension: {question: score (1-5)}}.
        """
        self.organization_name = organization_name
        self.scores = scores
        self._validate_scores()

    def _validate_scores(self):
        """Validate that all dimensions and questions are scored."""
        for dim_name, dim_data in ASSESSMENT_FRAMEWORK.items():
            if dim_name not in self.scores:
                raise ValueError(f"Missing dimension: {dim_name}")
            for q_name in dim_data["questions"]:
                if q_name not in self.scores[dim_name]:
                    raise ValueError(f"Missing question: {dim_name} > {q_name}")
                score = self.scores[dim_name][q_name]
                if not (1 <= score <= 5):
                    raise ValueError(f"Score out of range (1-5): {dim_name} > {q_name} = {score}")

    def dimension_averages(self) -> Dict[str, float]:
        """Calculate the average score for each dimension."""
        averages = {}
        for dim_name in ASSESSMENT_FRAMEWORK:
            dim_scores = list(self.scores[dim_name].values())
            averages[dim_name] = sum(dim_scores) / len(dim_scores)
        return averages

    def overall_score(self) -> float:
        """Calculate the weighted overall readiness score."""
        averages = self.dimension_averages()
        weighted_sum = sum(
            averages[dim] * ASSESSMENT_FRAMEWORK[dim]["weight"]
            for dim in ASSESSMENT_FRAMEWORK
        )
        return round(weighted_sum, 2)

    def readiness_level(self) -> str:
        """Classify overall readiness level."""
        score = self.overall_score()
        for (low, high), level in self.LEVELS.items():
            if low <= score < high:
                return level
        return "Unknown"

    def radar_chart_text(self) -> str:
        """Generate a text-based radar chart representation."""
        averages = self.dimension_averages()
        max_bar = 30  # characters for max score of 5

        lines = ["\nReadiness Radar (text visualization)\n"]
        for dim, avg in averages.items():
            bar_len = int((avg / 5) * max_bar)
            bar = "█" * bar_len + "░" * (max_bar - bar_len)
            lines.append(f"  {dim:<28s} {bar} {avg:.1f}/5")
        return "\n".join(lines)

    def strengths_and_gaps(self) -> Tuple[List[Tuple[str, str, int]], List[Tuple[str, str, int]]]:
        """Identify top 3 strengths and top 3 gaps (lowest scores)."""
        all_scores = []
        for dim_name in ASSESSMENT_FRAMEWORK:
            for q_name, score in self.scores[dim_name].items():
                all_scores.append((dim_name, q_name, score))

        # Sort by score
        sorted_asc = sorted(all_scores, key=lambda x: x[2])
        sorted_desc = sorted(all_scores, key=lambda x: x[2], reverse=True)

        gaps = sorted_asc[:3]
        strengths = sorted_desc[:3]

        return strengths, gaps

    def action_plan(self) -> List[Dict]:
        """Generate a prioritized action plan based on gaps."""
        averages = self.dimension_averages()
        # Sort dimensions by score (lowest first = highest priority)
        sorted_dims = sorted(averages.items(), key=lambda x: x[1])

        plan = []
        priority = 1
        for dim_name, avg_score in sorted_dims:
            if avg_score >= 4.0:
                actions = self.ACTION_PLANS.get(dim_name, {}).get("high", [])
                timeline = "Ongoing optimization"
            else:
                actions = self.ACTION_PLANS.get(dim_name, {}).get("low", [])
                if avg_score < 2.0:
                    timeline = "Immediate (0-3 months)"
                elif avg_score < 3.0:
                    timeline = "Short-term (3-6 months)"
                else:
                    timeline = "Medium-term (6-12 months)"

            plan.append({
                "priority": priority,
                "dimension": dim_name,
                "current_score": round(avg_score, 1),
                "timeline": timeline,
                "actions": actions,
            })
            priority += 1

        return plan

    def recommended_pattern(self) -> Dict[str, str]:
        """Recommend an integration pattern from Ch09 based on readiness."""
        score = self.overall_score()
        averages = self.dimension_averages()

        if score < 2.5:
            return {
                "pattern": "Sidecar",
                "description": (
                    "Start with AI as a sidecar — a helper alongside existing systems. "
                    "No changes to core architecture. Use AI for suggestions, summaries, "
                    "and insights that humans act on. This builds confidence with minimal risk."
                ),
                "examples": "Email drafting assistant, meeting summarizer, code review helper",
            }
        elif score < 3.5:
            return {
                "pattern": "Augmentation",
                "description": (
                    "You're ready for AI augmentation — embedding AI into existing workflows "
                    "to enhance human decision-making. AI handles routine cases; humans handle "
                    "exceptions. This is the highest-ROI pattern for most enterprises."
                ),
                "examples": "Claims triage (Ch14), document extraction, customer routing",
            }
        else:
            return {
                "pattern": "Rebuild",
                "description": (
                    "Your organization is ready to rebuild processes around AI as a "
                    "first-class component. Design new AI-native workflows rather than "
                    "retrofitting AI into old ones. This unlocks the largest gains but "
                    "requires the strongest foundation."
                ),
                "examples": "AI-native underwriting, autonomous operations, generative design",
            }

    def generate_report(self) -> str:
        """Generate the full readiness report."""
        lines = []

        # Header
        lines.append("=" * 70)
        lines.append(f"  AI READINESS ASSESSMENT REPORT")
        lines.append(f"  Organization: {self.organization_name}")
        lines.append("=" * 70)

        # Overall score
        score = self.overall_score()
        level = self.readiness_level()
        lines.append(f"\n  OVERALL READINESS SCORE: {score:.1f} / 5.0")
        lines.append(f"  READINESS LEVEL: {level.upper()}")

        # Radar chart
        lines.append(self.radar_chart_text())

        # Dimension detail
        lines.append("\n" + "-" * 70)
        lines.append("  DIMENSION SCORES")
        lines.append("-" * 70)

        averages = self.dimension_averages()
        dim_rows = []
        for dim_name, avg in averages.items():
            weight = ASSESSMENT_FRAMEWORK[dim_name]["weight"]
            dim_rows.append([
                dim_name,
                f"{avg:.1f}",
                f"{weight*100:.0f}%",
                f"{avg * weight:.2f}",
            ])
        lines.append(tabulate(
            dim_rows,
            headers=["Dimension", "Score", "Weight", "Weighted"],
            tablefmt="grid"
        ))

        # Sub-question detail
        lines.append("\n" + "-" * 70)
        lines.append("  DETAILED SCORES")
        lines.append("-" * 70)

        detail_rows = []
        for dim_name in ASSESSMENT_FRAMEWORK:
            for q_name, q_score in self.scores[dim_name].items():
                detail_rows.append([dim_name[:20], q_name, q_score, "█" * q_score + "░" * (5 - q_score)])
        lines.append(tabulate(
            detail_rows,
            headers=["Dimension", "Sub-Question", "Score", "Visual"],
            tablefmt="grid"
        ))

        # Strengths and gaps
        strengths, gaps = self.strengths_and_gaps()

        lines.append("\n" + "-" * 70)
        lines.append("  TOP 3 STRENGTHS")
        lines.append("-" * 70)
        for dim, question, score in strengths:
            lines.append(f"  [{score}/5] {dim} > {question}")

        lines.append("\n" + "-" * 70)
        lines.append("  TOP 3 GAPS (Priority Areas)")
        lines.append("-" * 70)
        for dim, question, score in gaps:
            rubric = ASSESSMENT_FRAMEWORK[dim]["questions"][question]["rubric"]
            current_desc = rubric.get(score, "")
            next_desc = rubric.get(min(score + 1, 5), "")
            lines.append(f"  [{score}/5] {dim} > {question}")
            lines.append(f"         Current: {current_desc}")
            lines.append(f"         Next level: {next_desc}")

        # Action plan
        plan = self.action_plan()
        lines.append("\n" + "-" * 70)
        lines.append("  PRIORITIZED ACTION PLAN")
        lines.append("-" * 70)

        for item in plan:
            lines.append(f"\n  Priority #{item['priority']}: {item['dimension']} "
                         f"(current: {item['current_score']}/5)")
            lines.append(f"  Timeline: {item['timeline']}")
            lines.append(f"  Actions:")
            for action in item["actions"]:
                lines.append(f"    -> {action}")

        # Recommended pattern
        pattern = self.recommended_pattern()
        lines.append("\n" + "-" * 70)
        lines.append("  RECOMMENDED INTEGRATION PATTERN (from Ch09)")
        lines.append("-" * 70)
        lines.append(f"  Pattern: {pattern['pattern'].upper()}")
        lines.append(f"  {pattern['description']}")
        lines.append(f"  Examples: {pattern['examples']}")

        lines.append("\n" + "=" * 70)

        return "\n".join(lines)

## Sample Assessment: A Typical Enterprise

This represents a mid-sized enterprise that has invested in cloud and data
engineering but lags in AI-specific governance and team skills. The mix of
2s, 3s, and 4s is realistic — very few organizations score uniformly.

In [ ]:
# Sample scores for a "typical enterprise"
# A mix of 2s, 3s, and 4s — realistic for a company beginning its AI journey

sample_scores = {
    "Data Maturity": {
        "Data Quality": 3,
        "Data Accessibility": 3,
        "Data Cataloging": 2,
        "Data Governance": 3,
    },
    "Technical Infrastructure": {
        "Cloud Adoption": 4,
        "API Availability": 3,
        "CI/CD Maturity": 4,
        "Compute Resources": 2,
    },
    "Team Skills": {
        "ML/AI Knowledge": 2,
        "Python Proficiency": 3,
        "Data Engineering": 3,
        "Prompt Engineering": 2,
    },
    "Governance": {
        "Data Governance Framework": 3,
        "Security Policies": 2,
        "Compliance Processes": 2,
        "Risk Management": 2,
    },
    "Organizational Readiness": {
        "Executive Sponsorship": 4,
        "Change Management": 3,
        "AI Strategy": 3,
        "Innovation Culture": 3,
    },
}

print("Sample scores loaded for 'Acme Financial Services'.")

## Generate the Readiness Report

Run the assessment and display the full report.

In [ ]:
assessment = AIReadinessAssessment(
    organization_name="Acme Financial Services",
    scores=sample_scores,
)

report = assessment.generate_report()
print(report)

## Interactive: Score Your Own Organization

Use the scoring rubrics above to rate your enterprise on each sub-question.
Replace the scores below with your honest assessment, then run the cell to
generate your personalized report.

**Tip:** Be honest. Overscoring leads to overambitious AI projects that fail.
Underscoring misses opportunities. Have 3-5 people score independently and
average the results for the most accurate picture.

In [ ]:
# ============================================================
# YOUR ASSESSMENT — edit the scores (1-5 for each)
# ============================================================

my_scores = {
    "Data Maturity": {
        "Data Quality": 3,           # How clean is your data?
        "Data Accessibility": 3,     # Can teams access what they need?
        "Data Cataloging": 2,        # Is data discoverable?
        "Data Governance": 2,        # Are there owners and policies?
    },
    "Technical Infrastructure": {
        "Cloud Adoption": 3,         # How cloud-mature are you?
        "API Availability": 2,       # Are systems API-accessible?
        "CI/CD Maturity": 3,         # How automated is deployment?
        "Compute Resources": 2,      # Can you get GPUs when needed?
    },
    "Team Skills": {
        "ML/AI Knowledge": 2,        # Does the team understand ML/AI?
        "Python Proficiency": 2,     # How strong is Python skill?
        "Data Engineering": 3,       # Can you build data pipelines?
        "Prompt Engineering": 1,     # Can you design effective prompts?
    },
    "Governance": {
        "Data Governance Framework": 2,  # Formal governance?
        "Security Policies": 2,          # AI-specific security?
        "Compliance Processes": 2,       # Can you prove compliance?
        "Risk Management": 1,            # AI risk assessment?
    },
    "Organizational Readiness": {
        "Executive Sponsorship": 3,   # Is there a champion?
        "Change Management": 2,       # Can you manage the change?
        "AI Strategy": 2,             # Is there a strategy?
        "Innovation Culture": 3,      # Is experimentation safe?
    },
}

my_assessment = AIReadinessAssessment(
    organization_name="My Organization",  # <-- Change this
    scores=my_scores,
)

print(my_assessment.generate_report())

## Display the Scoring Rubrics

Use this reference when scoring your organization. Print the full rubric
for each dimension so you can score accurately.

In [ ]:
# Print all rubrics for reference
for dim_name, dim_data in ASSESSMENT_FRAMEWORK.items():
    print(f"\n{'='*60}")
    print(f"  {dim_name} (weight: {dim_data['weight']*100:.0f}%)")
    print(f"{'='*60}")

    for q_name, q_data in dim_data["questions"].items():
        print(f"\n  {q_name}")
        print(f"  Question: {q_data['prompt']}")
        print(f"  Scoring rubric:")
        for score, description in q_data["rubric"].items():
            print(f"    {score}: {description}")

## Key Takeaways

1. **Honest self-assessment is the foundation** of any AI strategy. Skipping
   this step is the single biggest predictor of AI project failure.

2. **Data maturity carries the highest weight** because even the best models
   can't compensate for poor data. Fix data first.

3. **Governance gaps are the hardest to close** and the most dangerous to
   ignore — especially in regulated industries.

4. **Match your integration pattern to your readiness level.** A sidecar
   pattern for a low-readiness org delivers value without creating risk.
   Don't rebuild when you should be augmenting.

5. **Reassess quarterly.** Readiness changes as you invest. What was a gap
   three months ago may be a strength today.

Use this assessment as a conversation starter with your leadership team.
The gaps identified here become the roadmap for your AI transformation.